In [5]:
import pandas as pd
df = pd.read_csv("archive/medical_examination.csv")
print("資料維度:", df.shape)
# 查看前幾筆資料
df.head()

資料維度: (70000, 13)


,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,0,18393,2,168,62.0,110,80,1,1,0,0,1,0
1,1,20228,1,156,85.0,140,90,3,1,0,0,1,1
2,2,18857,1,165,64.0,130,70,3,1,0,0,0,1
3,3,17623,2,169,82.0,150,100,1,1,0,0,1,1
4,4,17474,1,156,56.0,100,60,1,1,0,0,0,0


In [6]:
import matplotlib.pyplot as plt
import seaborn as sns
# 設定字型為支援中文的字型（以繁體中文常用字型為例）
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei']  # 微軟正黑體
plt.rcParams['axes.unicode_minus'] = False  # 讓負號正常顯示

# 檢查資料結構與缺值
print(df.info())
print(df.isnull().sum())

# 加入 age_years 欄位（將 age 從天轉換成年）
df['age_years'] = (df['age'] / 365).astype(int)

# 移除不合理的血壓值
df = df[(df['ap_hi'] > df['ap_lo']) & (df['ap_hi'] > 0) & (df['ap_lo'] > 0)]

# 去除 gender 中的錯誤值
df = df[df['gender'].isin([1, 2])]

# 使用 IQR（四分位距）法去除離群值
cols = ['height', 'weight', 'ap_hi', 'ap_lo']

for col in cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df = df[(df[col] >= lower) & (df[col] <= upper)]

print("去除離群值後資料筆數：", df.shape)

# 去除重複值
df = df.drop_duplicates()
# ---------- 移除 'age' 和 'id' 欄位 ----------
if 'age' in df.columns:
    df = df.drop(columns=['age'])
if 'id' in df.columns:
    df = df.drop(columns=['id'])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70000 entries, 0 to 69999
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           70000 non-null  int64  
 1   age          70000 non-null  int64  
 2   gender       70000 non-null  int64  
 3   height       70000 non-null  int64  
 4   weight       70000 non-null  float64
 5   ap_hi        70000 non-null  int64  
 6   ap_lo        70000 non-null  int64  
 7   cholesterol  70000 non-null  int64  
 8   gluc         70000 non-null  int64  
 9   smoke        70000 non-null  int64  
 10  alco         70000 non-null  int64  
 11  active       70000 non-null  int64  
 12  cardio       70000 non-null  int64  
dtypes: float64(1), int64(12)
memory usage: 6.9 MB
None
id             0
age            0
gender         0
height         0
weight         0
ap_hi          0
ap_lo          0
cholesterol    0
gluc           0
smoke          0
alco           0
active         0
c

In [7]:
# ----------------------------------------------------
# 匯入套件
# ----------------------------------------------------
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.pipeline import Pipeline
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.svm import SVC
import numpy as np

# ----------------------------------------------------
# 資料（已前處理完畢）
# ----------------------------------------------------
# df 為清理後的資料集
X = df.drop(columns=['cardio'])
y = df['cardio']

# 數值與類別欄位
num_cols = ['age_years', 'height', 'weight', 'ap_hi', 'ap_lo']
cat_cols = ['gender', 'cholesterol', 'gluc', 'smoke', 'alco', 'active']

# ----------------------------------------------------
# 建立不同的前處理器
# ----------------------------------------------------
# 標準化 + OneHot
pre_std_onehot = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

# 正規化 + OneHot
pre_minmax_onehot = ColumnTransformer([
    ('num', MinMaxScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

# 標準化（無 OneHot）
pre_std = ColumnTransformer([
    ('num', StandardScaler(), num_cols)
])

# 正規化（無 OneHot）
pre_minmax = ColumnTransformer([
    ('num', MinMaxScaler(), num_cols)
])

# ----------------------------------------------------
# 超參數搜尋空間
# ----------------------------------------------------
param_grid = {
    'classifier__n_neighbors': list(range(1, 21)),
    'classifier__weights': ['uniform', 'distance'],
    'classifier__p': [1, 2]
}

#### 標準化

In [8]:
from sklearn.preprocessing import StandardScaler

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols)
], remainder='passthrough')

split_size = len(df) // 3
results_std = []

for i in range(3):
    print(f"\n==== (標準化) 第 {i+1} 次資料分割與訓練 ====")
    start, end = i * split_size, (i + 1) * split_size if i < 2 else len(df)
    df_sub = df.iloc[start:end]
    X_sub, y_sub = df_sub.drop(columns=['cardio']), df_sub['cardio']

    X_train, X_test, y_train, y_test = train_test_split(X_sub, y_sub, test_size=0.2, stratify=y_sub, random_state=42+i)

    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', SVC(probability=True))
    ])

    param_grid = {
        'classifier__C': [0.1, 1, 10],
        'classifier__kernel': ['linear', 'rbf'],
        'classifier__gamma': ['scale', 'auto']
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42+i)
    grid = GridSearchCV(pipe, param_grid, cv=cv, scoring='accuracy', n_jobs=-1)
    grid.fit(X_train, y_train)

    print("最佳參數:", grid.best_params_)
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    print(f"第{i+1}次結果：Accuracy={acc:.4f}, Precision={prec:.4f}, Recall={rec:.4f}, F1={f1:.4f}, AUC={auc:.4f}")
    results_std.append([acc, prec, rec, f1, auc])

df_std = pd.DataFrame(results_std, columns=['Accuracy','Precision','Recall','F1','AUC'])

print("\n=============================")
print("📊 SVM 三等份實驗結果")
print(df_std)
print("\n平均結果：\n", df_std.mean().to_frame().T)


==== (標準化) 第 1 次資料分割與訓練 ====
最佳參數: {'classifier__C': 1, 'classifier__gamma': 'auto', 'classifier__kernel': 'rbf'}
第1次結果：Accuracy=0.7405, Precision=0.7702, Recall=0.6685, F1=0.7158, AUC=0.7933

==== (標準化) 第 2 次資料分割與訓練 ====
最佳參數: {'classifier__C': 1, 'classifier__gamma': 'auto', 'classifier__kernel': 'rbf'}
第2次結果：Accuracy=0.7302, Precision=0.7626, Recall=0.6639, F1=0.7099, AUC=0.7886

==== (標準化) 第 3 次資料分割與訓練 ====
最佳參數: {'classifier__C': 10, 'classifier__gamma': 'scale', 'classifier__kernel': 'rbf'}
第3次結果：Accuracy=0.7250, Precision=0.7541, Recall=0.6607, F1=0.7043, AUC=0.7771

📊 SVM 三等份實驗結果
   Accuracy  Precision    Recall        F1       AUC
0  0.740518   0.770232  0.668468  0.715751  0.793308
1  0.730197   0.762618  0.663930  0.709861  0.788568
2  0.724982   0.754144  0.660697  0.704334  0.777139

平均結果：
    Accuracy  Precision    Recall        F1       AUC
0  0.731899   0.762331  0.664365  0.709982  0.786338


#### 標準化+one hot encodeing

In [ ]:
# =============================
# 1️⃣ 資料前處理
# =============================
X = df.drop(columns=['cardio'])
y = df['cardio']

num_cols = ['age_years', 'height', 'weight', 'ap_hi', 'ap_lo']
cat_cols = ['gender', 'cholesterol', 'gluc', 'smoke', 'alco', 'active']

# 建立前處理管線
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(), cat_cols)
])

# =============================
# 2️⃣ 三等分資料集
# =============================
split_size = len(df) // 3
results = []

for i in range(3):
    print(f"\n==== 第 {i+1} 次資料分割與訓練 ====")
    
    # 切出該次的子集
    start_idx = i * split_size
    end_idx = (i + 1) * split_size if i < 2 else len(df)
    df_subset = df.iloc[start_idx:end_idx]

    X_sub = df_subset.drop(columns=['cardio'])
    y_sub = df_subset['cardio']

    # 分割訓練與測試集 (8:2)
    X_train, X_test, y_train, y_test = train_test_split(
        X_sub, y_sub, test_size=0.2, stratify=y_sub, random_state=42+i
    )

    # =============================
    # 3️⃣ 定義 SVM + 調參範圍
    # =============================
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', SVC(probability=True))
    ])

    param_grid = {
        'classifier__C': [0.1, 1, 10],
        'classifier__kernel': ['linear', 'rbf'],
        'classifier__gamma': ['scale', 'auto']
    }

    # K-Fold 設定
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42+i)

    grid = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        cv=cv,
        scoring='accuracy',
        n_jobs=-1
    )

    # =============================
    # 4️⃣ 模型訓練與最佳參數
    # =============================
    grid.fit(X_train, y_train)
    print("最佳參數:", grid.best_params_)

    # =============================
    # 5️⃣ 模型評估
    # =============================
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    print(f"第{i+1}次結果：Accuracy={acc:.4f}, Precision={prec:.4f}, Recall={rec:.4f}, F1={f1:.4f}, AUC={auc:.4f}")
    results.append([acc, prec, rec, f1, auc])

# =============================
# 6️⃣ 統計三次平均結果
# =============================
results_df = pd.DataFrame(results, columns=['Accuracy', 'Precision', 'Recall', 'F1', 'AUC'])
avg = results_df.mean().to_frame().T
avg.index = ['三次平均']

print("\n=============================")
print("📊 SVM 三等份實驗結果")
print(results_df)
print("\n平均結果：")
print(avg)



==== 第 1 次資料分割與訓練 ====
最佳參數: {'classifier__C': 1, 'classifier__gamma': 'scale', 'classifier__kernel': 'rbf'}
第1次結果：Accuracy=0.7432, Precision=0.7747, Recall=0.6690, F1=0.7180, AUC=0.7923

==== 第 2 次資料分割與訓練 ====
最佳參數: {'classifier__C': 0.1, 'classifier__gamma': 'auto', 'classifier__kernel': 'rbf'}
第2次結果：Accuracy=0.7273, Precision=0.7682, Recall=0.6465, F1=0.7021, AUC=0.7902

==== 第 3 次資料分割與訓練 ====
最佳參數: {'classifier__C': 1, 'classifier__gamma': 'scale', 'classifier__kernel': 'rbf'}
第3次結果：Accuracy=0.7238, Precision=0.7552, Recall=0.6554, F1=0.7017, AUC=0.7819

📊 SVM 三等份實驗結果
   Accuracy  Precision    Recall        F1       AUC
0  0.743159   0.774744  0.668959  0.717976  0.792314
1  0.727316   0.768216  0.646548  0.702150  0.790249
2  0.723782   0.755159  0.655373  0.701736  0.781869

平均結果：
      Accuracy  Precision   Recall        F1       AUC
三次平均  0.731419    0.76604  0.65696  0.707287  0.788144


#### 正規化

In [10]:
from sklearn.preprocessing import MinMaxScaler

preprocessor = ColumnTransformer([
    ('num', MinMaxScaler(), num_cols)
], remainder='passthrough')  # 類別欄位直接保留為數值

split_size = len(df) // 3
results_norm = []

for i in range(3):
    print(f"\n==== (正規化) 第 {i+1} 次資料分割與訓練 ====")
    start, end = i * split_size, (i + 1) * split_size if i < 2 else len(df)
    df_sub = df.iloc[start:end]
    X_sub, y_sub = df_sub.drop(columns=['cardio']), df_sub['cardio']

    X_train, X_test, y_train, y_test = train_test_split(X_sub, y_sub, test_size=0.2, stratify=y_sub, random_state=42+i)

    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', SVC(probability=True))
    ])

    param_grid = {
        'classifier__C': [0.1, 1, 10],
        'classifier__kernel': ['linear', 'rbf'],
        'classifier__gamma': ['scale', 'auto']
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42+i)
    grid = GridSearchCV(pipe, param_grid, cv=cv, scoring='accuracy', n_jobs=-1)
    grid.fit(X_train, y_train)

    print("最佳參數:", grid.best_params_)
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    print(f"第{i+1}次結果：Accuracy={acc:.4f}, Precision={prec:.4f}, Recall={rec:.4f}, F1={f1:.4f}, AUC={auc:.4f}")
    results_norm.append([acc, prec, rec, f1, auc])

df_norm = pd.DataFrame(results_norm, columns=['Accuracy','Precision','Recall','F1','AUC'])
print("\n=============================")
print("📊 SVM 三等份實驗結果")
print(df_norm)
print("\n平均結果：\n", df_norm.mean().to_frame().T)


==== (正規化) 第 1 次資料分割與訓練 ====
最佳參數: {'classifier__C': 10, 'classifier__gamma': 'auto', 'classifier__kernel': 'rbf'}
第1次結果：Accuracy=0.7412, Precision=0.7838, Recall=0.6498, F1=0.7105, AUC=0.7906

==== (正規化) 第 2 次資料分割與訓練 ====
最佳參數: {'classifier__C': 10, 'classifier__gamma': 'auto', 'classifier__kernel': 'rbf'}
第2次結果：Accuracy=0.7252, Precision=0.7619, Recall=0.6504, F1=0.7017, AUC=0.7885

==== (正規化) 第 3 次資料分割與訓練 ====
最佳參數: {'classifier__C': 10, 'classifier__gamma': 'scale', 'classifier__kernel': 'rbf'}
第3次結果：Accuracy=0.7274, Precision=0.7633, Recall=0.6525, F1=0.7035, AUC=0.7848

📊 SVM 三等份實驗結果
   Accuracy  Precision    Recall        F1       AUC
0  0.741239   0.783768  0.649804  0.710526  0.790591
1  0.725156   0.761878  0.650410  0.701745  0.788531
2  0.727382   0.763307  0.652469  0.703549  0.784846

平均結果：
    Accuracy  Precision    Recall        F1      AUC
0  0.731259   0.769651  0.650894  0.705274  0.78799


#### 正規化+one hot encoding

In [9]:
# ===== 1. 定義資料 =====
X = df.drop(columns=['cardio'])
y = df['cardio']

num_cols = ['age_years', 'height', 'weight', 'ap_hi', 'ap_lo']
cat_cols = ['gender', 'cholesterol', 'gluc', 'smoke', 'alco', 'active']

# ===== 2. 正規化 + One-Hot =====
preprocessor = ColumnTransformer([
    ('num', MinMaxScaler(), num_cols),
    ('cat', OneHotEncoder(), cat_cols)
])

# ===== 3. 三等分實驗 =====
split_size = len(df) // 3
results_norm_onehot = []

for i in range(3):
    print(f"\n==== (正規化+OneHot) 第 {i+1} 次資料分割與訓練 ====")
    start, end = i * split_size, (i + 1) * split_size if i < 2 else len(df)
    df_sub = df.iloc[start:end]
    X_sub, y_sub = df_sub.drop(columns=['cardio']), df_sub['cardio']

    X_train, X_test, y_train, y_test = train_test_split(X_sub, y_sub, test_size=0.2, stratify=y_sub, random_state=42+i)

    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', SVC(probability=True))
    ])

    param_grid = {
        'classifier__C': [0.1, 1, 10],
        'classifier__kernel': ['linear', 'rbf'],
        'classifier__gamma': ['scale', 'auto']
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42+i)
    grid = GridSearchCV(pipe, param_grid, cv=cv, scoring='accuracy', n_jobs=-1)
    grid.fit(X_train, y_train)

    print("最佳參數:", grid.best_params_)
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    print(f"第{i+1}次結果：Accuracy={acc:.4f}, Precision={prec:.4f}, Recall={rec:.4f}, F1={f1:.4f}, AUC={auc:.4f}")
    results_norm_onehot.append([acc, prec, rec, f1, auc])

df_norm_onehot = pd.DataFrame(results_norm_onehot, columns=['Accuracy','Precision','Recall','F1','AUC'])
print("\n=============================")
print("📊 SVM 三等份實驗結果")
print(df_norm_onehot)
print("\n平均結果：\n", df_norm_onehot.mean().to_frame().T)



==== (正規化+OneHot) 第 1 次資料分割與訓練 ====
最佳參數: {'classifier__C': 1, 'classifier__gamma': 'auto', 'classifier__kernel': 'rbf'}
第1次結果：Accuracy=0.7388, Precision=0.7862, Recall=0.6395, F1=0.7053, AUC=0.7910

==== (正規化+OneHot) 第 2 次資料分割與訓練 ====
最佳參數: {'classifier__C': 1, 'classifier__gamma': 'scale', 'classifier__kernel': 'rbf'}
第2次結果：Accuracy=0.7256, Precision=0.7658, Recall=0.6456, F1=0.7006, AUC=0.7876

==== (正規化+OneHot) 第 3 次資料分割與訓練 ====
最佳參數: {'classifier__C': 1, 'classifier__gamma': 'auto', 'classifier__kernel': 'rbf'}
第3次結果：Accuracy=0.7269, Precision=0.7710, Recall=0.6389, F1=0.6988, AUC=0.7851

📊 SVM 三等份實驗結果
   Accuracy  Precision    Recall        F1       AUC
0  0.738838   0.786232  0.639489  0.705309  0.790996
1  0.725636   0.765750  0.645582  0.700550  0.787616
2  0.726902   0.771028  0.638916  0.698782  0.785146

平均結果：
    Accuracy  Precision    Recall        F1       AUC
0  0.730459   0.774337  0.641329  0.701547  0.787919
